In [2]:
import pandas as pd
import numpy as np

# Semilla para reproducibilidad
np.random.seed(42)
n_samples = 947  # Número de clientes

# ID de cliente
customer_id = np.arange(1, n_samples + 1)

# Número de niños por cliente
num_kids_per_customer = np.random.choice([1, 2, 3], size=n_samples, p=[0.7, 0.2, 0.1])

# Expandir datos a nivel niño
kid_ids = []
customer_expanded = []
for i, num_kids in enumerate(num_kids_per_customer):
    for j in range(num_kids):
        kid_ids.append(f"{i+1}_{j+1}")
        customer_expanded.append(i + 1)
total_kids = len(kid_ids)

# Edad del niño
kids_age = np.clip(np.random.normal(9, 2, size=total_kids).astype(int), 4, 14)

# Sexo del niño
kids_sex = np.random.choice([0, 1], size=total_kids, p=[0.85, 0.15])

# Ingreso familiar
family_income = np.random.choice(
    [25000, 50000, 75000, 100000, 150000, 200000],
    size=n_samples,
    p=[0.1, 0.2, 0.3, 0.25, 0.1, 0.05]
)
family_income_expanded = np.array([family_income[cust-1] for cust in customer_expanded], dtype=float)

# Canal de adquisición
signup_channels = []
for income in family_income:
    if income >= 150000:
        signup_channels.append(np.random.choice(['Referral', 'Organic'], p=[0.7, 0.3]))
    elif income >= 75000:
        signup_channels.append(np.random.choice(['Referral', 'Organic', 'Ads'], p=[0.4, 0.4, 0.2]))
    else:
        signup_channels.append(np.random.choice(['Ads', 'Social Media'], p=[0.6, 0.4]))
signup_channel_expanded = [signup_channels[cust-1] for cust in customer_expanded]

# Tipo de suscripción
subscription_type = []
for i in range(n_samples):
    if num_kids_per_customer[i] > 1:
        subscription_type.append("Familiar")
    else:
        if family_income[i] >= 150000:
            subscription_type.append("Premium")
        elif family_income[i] >= 75000:
            subscription_type.append(np.random.choice(["Premium", "Básica"], p=[0.7, 0.3]))
        elif family_income[i] >= 40000:
            subscription_type.append(np.random.choice(["Premium", "Básica"], p=[0.4, 0.6]))
        else:
            subscription_type.append(np.random.choice(["Premium", "Básica"], p=[0.1, 0.9]))
subscription_type_expanded = [subscription_type[cust-1] for cust in customer_expanded]

# Tarifa mensual
monthly_fee_dict = {'Básica': 10, 'Premium': 20, 'Familiar': 35}
monthly_fee_expanded = np.array([monthly_fee_dict[sub] for sub in subscription_type_expanded], dtype=float)
for i in range(n_samples):
    if subscription_type[i] == "Familiar":
        num_kids = num_kids_per_customer[i]
        monthly_fee_expanded[np.where(np.array(customer_expanded) == (i + 1))] = 35 / num_kids

# Tiempo de sesión promedio (según tipo de suscripción y sexo del niño)
avg_session_time = (
    (monthly_fee_expanded * 1.2) +
    np.where(kids_sex == 0, np.random.normal(5, 2, total_kids), np.random.normal(2, 1, total_kids))
)

# Tickets de soporte (más en básica y con menos ingreso)
base_tickets = np.where(
    np.array([subscription_type_expanded[i-1] for i in customer_expanded]) == "Básica",
    np.random.poisson(3, total_kids),
    np.random.poisson(1, total_kids)
)
customer_support_tickets_expanded = base_tickets + (family_income_expanded < 50000).astype(int)

# Tiempo de resolución de tickets
issue_resolution_time = (
    np.random.normal(3, 1, size=total_kids) +
    customer_support_tickets_expanded * 0.5 -
    (avg_session_time > 45).astype(int)
)

# Churn con reglas más marcadas
churn_probs = (
    np.array([0.6 if sub == 'Básica' else 0.25 if sub == 'Premium' else 0.08 for sub in subscription_type_expanded]) +
    customer_support_tickets_expanded * 0.08 +
    issue_resolution_time * 0.02 -
    avg_session_time * 0.005
)
churn_probs = np.clip(churn_probs, 0, 1)
churn = np.random.binomial(1, churn_probs)

# Tenure
base_tenure = (
    5 +
    (avg_session_time / 8) -
    customer_support_tickets_expanded * 0.4 +
    np.random.normal(0, 2, size=total_kids)
)
tenure = np.clip(base_tenure, 1, 36).astype(int)
tenure = np.where(churn == 1, np.clip(np.random.randint(1, tenure + 1), 1, 36), tenure)

# Total de sesiones
avg_sessions_per_week = np.where(avg_session_time > 40, 5, 3)
total_sessions = (tenure * avg_sessions_per_week * 4).astype(int)

# Crear DataFrame
df = pd.DataFrame({
    'customer_id': customer_expanded,
    'kid_id': kid_ids,
    'kids_age': kids_age,
    'kids_sex': kids_sex,
    'family_income': family_income_expanded,
    'subscription_type': subscription_type_expanded,
    'monthly_fee': monthly_fee_expanded,
    'avg_session_time': avg_session_time,
    'customer_support_tickets': customer_support_tickets_expanded,
    'issue_resolution_time': issue_resolution_time,
    'signup_channel': signup_channel_expanded,
    'churn': churn,
    'tenure': tenure,
    'total_sessions': total_sessions
})

In [3]:
# Introducir datos nulos y outliers al dataset para simular casos reales

df_dirty = df.copy()

# Insertar nulos aleatorios (5% de cada columna objetivo)
cols_with_nans = ['kids_age', 'kids_sex', 'family_income', 'avg_session_time']
for col in cols_with_nans:
    idx = df_dirty.sample(frac=0.05, random_state=42).index
    df_dirty.loc[idx, col] = np.nan

# Introducir outliers intencionales
# - kids_age fuera del rango (ej: 2 y 25 años)
outlier_indices_age = df_dirty.sample(frac=0.01, random_state=1).index
df_dirty.loc[outlier_indices_age, 'kids_age'] = np.random.choice([2, 25], size=len(outlier_indices_age))

# - kids_sex con valores no válidos (ej: 2, 3)
outlier_indices_sex = df_dirty.sample(frac=0.01, random_state=2).index
df_dirty.loc[outlier_indices_sex, 'kids_sex'] = np.random.choice([2, 3], size=len(outlier_indices_sex))

# - family_income exageradamente alto
outlier_indices_income = df_dirty.sample(frac=0.01, random_state=3).index
df_dirty.loc[outlier_indices_income, 'family_income'] = df_dirty['family_income'].max() * 5

# - avg_session_time con valores excesivos
outlier_indices_session = df_dirty.sample(frac=0.01, random_state=4).index
df_dirty.loc[outlier_indices_session, 'avg_session_time'] = df_dirty['avg_session_time'].max() * 4

In [ ]:
df_dirty.to_csv("data/gabu_dataset_2.csv")